In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "app").exists():
            return candidate
    raise FileNotFoundError("Could not locate the BTS Song Atlas project root.")


PROJECT_ROOT = find_project_root()


In [ ]:
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
PERSONAL_DIR = DATA_PROCESSED / "personal_atlas"

overlay = pd.read_parquet(PERSONAL_DIR / "personal_atlas_overlay.parquet")
song_league = pd.read_parquet(PERSONAL_DIR / "bts_song_league.parquet")
album_league = pd.read_parquet(PERSONAL_DIR / "bts_album_league.parquet")

overlay.shape, song_league.shape, album_league.shape

In [ ]:
top_songs = (
    overlay[overlay["personal_hours"] > 0]
    .sort_values("personal_hours", ascending=False)
    [["track_name", "album_name", "personal_hours", "personal_plays", "mastery_level"]]
    .head(20)
)

top_songs

In [ ]:
mastery_distribution = (
    overlay
    .groupby("mastery_level")
    .agg(
        songs=("track_name", "count"),
        total_hours=("personal_hours", "sum"),
    )
    .reset_index()
)

mastery_distribution

In [ ]:
fig = px.bar(
    mastery_distribution,
    x="mastery_level",
    y="songs",
    title="Personal mastery distribution",
    labels={
        "mastery_level": "Mastery level",
        "songs": "Number of songs",
    },
)

fig.show()

In [ ]:
top_albums = album_league.sort_values("hours_played", ascending=False).head(20)
top_albums

In [ ]:
fig = px.bar(
    top_albums.sort_values("hours_played"),
    x="hours_played",
    y="master_metadata_album_album_name",
    color="master_metadata_album_artist_name",
    orientation="h",
    title="Top BTS universe albums by listening hours",
    labels={
        "hours_played": "Hours played",
        "master_metadata_album_album_name": "Album",
        "master_metadata_album_artist_name": "Artist",
    },
)

fig.show()

In [ ]:
[col for col in overlay.columns if col.lower() in ["x", "y", "umap_x", "umap_y"] or "x" == col.lower() or "y" == col.lower()]

## Merge atlas coordinates

The Streamlit atlas loads `song_atlas.csv`, so this production export provides the existing two-dimensional semantic layout.

In [ ]:
ATLAS_PATH = DATA_PROCESSED / "song_atlas.csv"
atlas = pd.read_csv(ATLAS_PATH)

coordinate_columns = [
    col
    for col in ["track_id", "x", "y", "cluster", "canonical_title", "is_primary_version"]
    if col in atlas.columns
]

print("Shape:", atlas.shape)
print("Relevant columns:", coordinate_columns)
atlas[coordinate_columns].head()

In [ ]:
atlas_coordinate_check = pd.Series(
    {
        "rows": len(atlas),
        "unique_track_ids": atlas["track_id"].nunique(),
        "duplicate_track_ids": atlas["track_id"].duplicated().sum(),
        "missing_x": atlas["x"].isna().sum(),
        "missing_y": atlas["y"].isna().sum(),
    }
)

atlas_coordinate_check

In [ ]:
atlas_columns = [
    col
    for col in ["track_id", "x", "y", "cluster", "canonical_title", "is_primary_version"]
    if col in atlas.columns and (col == "track_id" or col not in overlay.columns)
]
atlas_coordinates = atlas[atlas_columns].copy()

merge_validation = (
    "one_to_one"
    if overlay["track_id"].is_unique and atlas_coordinates["track_id"].is_unique
    else "many_to_one"
)

personal_map = overlay.merge(
    atlas_coordinates,
    on="track_id",
    how="left",
    validate=merge_validation,
    indicator=True,
)

merge_validation, personal_map.shape

In [ ]:
valid_coordinates = personal_map[["x", "y"]].notna().all(axis=1)
unmatched = personal_map[personal_map["_merge"] == "left_only"]

merge_check = pd.Series(
    {
        "rows_before_merge": len(overlay),
        "rows_after_merge": len(personal_map),
        "coordinate_coverage_pct": valid_coordinates.mean() * 100,
        "unmatched_track_ids": unmatched["track_id"].nunique(),
        "duplicate_track_ids": personal_map["track_id"].duplicated().sum(),
        "rows_with_listening_history": personal_map["has_listening_history"].sum(),
        "rows_without_listening_history": (~personal_map["has_listening_history"]).sum(),
    }
)

merge_check

In [ ]:
unmatched[["track_id", "track_name", "album_name"]].head(10)

In [ ]:
personal_map = personal_map.drop(columns="_merge")

identity_columns = [
    "track_id",
    "track_name",
    "album_name",
    "release_year",
    "canonical_title",
    "is_primary_version",
    "x",
    "y",
    "cluster",
]
personal_columns = [
    "normalized_atlas_title",
    "personal_plays",
    "personal_hours",
    "personal_total_ms",
    "personal_first_play",
    "personal_last_play",
    "source_history_titles",
    "match_types",
    "min_fuzzy_score",
    "personal_rank",
    "mastery_level",
    "has_listening_history",
]
map_columns = [col for col in identity_columns + personal_columns if col in personal_map.columns]
personal_map = personal_map[map_columns].copy()

OUTPUT_PATH = PERSONAL_DIR / "personal_atlas_map.parquet"
personal_map.to_parquet(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)
print("Shape:", personal_map.shape)
print(f"Coordinate coverage: {valid_coordinates.mean():.1%}")

## Personal Atlas preview

In [ ]:
preview_map = personal_map.assign(
    preview_size=personal_map["personal_plays"].pow(0.5).add(4)
)

fig = px.scatter(
    preview_map,
    x="x",
    y="y",
    color="personal_hours",
    size="preview_size",
    size_max=18,
    hover_name="track_name",
    hover_data={
        "x": False,
        "y": False,
        "preview_size": False,
        "album_name": True,
        "personal_hours": ":.2f",
        "personal_plays": True,
        "mastery_level": True,
    },
    title="Personal Atlas: listening intensity across the semantic map",
    labels={
        "album_name": "Album",
        "personal_hours": "Hours",
        "personal_plays": "Plays",
        "mastery_level": "Mastery level",
    },
    template="plotly_dark",
)

fig.update_traces(marker={"opacity": 0.8, "line": {"width": 0}})
fig.update_layout(
    height=700,
    margin={"l": 0, "r": 0, "t": 60, "b": 0},
    plot_bgcolor="#060817",
    paper_bgcolor="#060817",
    coloraxis_colorbar={"title": "Listening hours"},
)
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False, scaleanchor="x", scaleratio=1)

fig.show()

## Conclusion

The production atlas coordinates are now joined to the personal listening overlay with complete merge coverage, and `personal_atlas_map.parquet` provides a visualization-ready dataset. The Personal Atlas can now support a heatmap or listening-intensity mode while preserving the semantic layout. Future reruns after the solo-discography expansion should improve personal-history coverage.